## 5.Entropy and OCP estimation

In this section we adapt the OCV estimation code to estimate open circuit potential (OCP) from half-cell charge/discharge, the process is esentially the same with minor changes, data conversion and estimation process can be found bellow.

### Half-cells experiment data conversion

Input the half-cell code name "ID" and temperatures charge/discharge experiments were performed on half-cells

In [ ]:
import pandas as pd
import numpy as np

cellIDs = ["LFPhc"]      # Identifiers for each cell
order = [25]  # Temperatures for each (°C)

headers = [
    "Step_Index",
    "Voltage(V)",
    "Charge_Capacity(mAh)",
    "Discharge_Capacity(mAh)"
]

# Corresponding field names to use internally
fields = ["step", "voltage", "chgmAh", "dismAh"]
column_map = dict(zip(headers, fields))

# Field names for the four different testing scripts
stepFields = ["script1", "script2", "script3", "script4"]

Bellow is the code to convert half-cell charge/discharge experiment:

In [ ]:
from pathlib import Path

for cellID in cellIDs:

    dirname = cellID.split("_")[0]  # remove "_" and everything after

    for temp in order:

        # Build file prefix
        if temp < 0:
            prefix = f"DATA/{dirname}_OCP/{cellID}_OCP_N{abs(temp):02d}"
        else:
            prefix = f"DATA/{dirname}_OCP/{cellID}_OCP_P{temp:02d}"

        data = {}

        # MATLAB used [1,3]
        for script in [1, 3]:

            file_path = Path(f"{prefix}_S{script}.xlsx") #file path, name and format

            if not file_path.exists():
                print(f"File not found: {file_path}")
                continue

            print(f"Reading {file_path}")
            #------------- Excel file processing -------------
            xls = pd.ExcelFile(file_path)
            OCVData = []

            for sheet in xls.sheet_names:
                if sheet == "Info":
                   continue

                print(f"Processing sheet {sheet}")
                df = pd.read_excel(file_path, sheet_name=sheet)

                df_filtered = df[headers].rename(columns=column_map)
                OCVData.append(df_filtered)
            #-------------------------------------------------

            #------------- CSV file processing ---------------
            #OCVData = []
            
            #print(f"Processing {file_path}")
            #df = pd.read_csv(file_path)

            #df_filtered = df[headers].rename(columns=column_map)
            #OCVData.append(df_filtered)
            #-------------------------------------------------

            if OCVData:
                combined_df = pd.concat(OCVData, ignore_index=True)

                # 🔹 Save directly next to original data file
                out_file = file_path.with_suffix(".pkl")
                combined_df.to_pickle(out_file)

                print(f"Saved {out_file}")

### Entropy and OCP estimation 

Similarly as to OCV estimation here we define the half-cell code name "ID", operating temepratures and operating voltage limits for ploting purposes.

In [ ]:
%reset -f

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Identifiers for each cell
cellIDs = ["LFPhc"]

# Available temperatures for each cell
temps = [[25]]

# Voltage limits for plotting
minV = [2.5]
maxV = [3.65]

Then we can run the OCP and entropy estimation algorithm to estimate OCP and entropy from half-cell charge/discharge datasets.

In [ ]:
import os
import pickle
from Functions.process_function_OCP import processOCP  # previously converted function


for theID, dirname in enumerate(cellIDs):
    cellID = dirname
    # Remove trailing underscore if present
    if '_' in dirname:
        dirname = dirname.split('_')[0]

    OCPDir = f"DATA/{dirname}_OCP"
    if not os.path.isdir(OCPDir):
        raise FileNotFoundError(
            f'Folder "{OCPDir}" not found in current folder.\n'
            f'Please change folders so that "{OCPDir}" is in the current folder and re-run.'
        )

    filetemps = np.array(temps[theID])
    numtemps = len(filetemps)
    data = [{} for _ in range(numtemps)]  # initialize data list

    # Load the OCP files
    for k, temp in enumerate(filetemps):

        if temp < 0:
            base = f"{cellID}_OCP_N{abs(temp):02d}"
        else:
            base = f"{cellID}_OCP_P{temp:02d}"

        file_s1 = os.path.join(OCPDir, f"{base}_S1.pkl")
        file_s3 = os.path.join(OCPDir, f"{base}_S3.pkl")

        if not os.path.isfile(file_s1):
            raise FileNotFoundError(f"Data file {file_s1} not found.")

        if not os.path.isfile(file_s3):
            raise FileNotFoundError(f"Data file {file_s3} not found.")

        with open(file_s1, 'rb') as f:
            script1_data = pickle.load(f)

        with open(file_s3, 'rb') as f:
            script3_data = pickle.load(f)

        data[k]['temp'] = temp
        data[k]['script1'] = script1_data
        data[k]['script3'] = script3_data

    # Call the processing function to generate OCP model
    model = processOCP(data, cellID, minV[theID], maxV[theID], savePlots=True)

    # Save model as a .pkl file
    os.makedirs('Models', exist_ok=True)
    model_filename = f"Models/{cellID}_model_ocp.pkl"
    with open(model_filename, 'wb') as f:
        pickle.dump(model, f)

    print(f"Processed model saved to {model_filename}")

Compilng the OCP and entropy estimation results into tables, and plot entropy profile.

In [ ]:
# Create OCP table
OCPTable = pd.DataFrame({
    "SOC": model["SOCaprox"],
    "OCP": model["OCVaprox"]
})

# Create entropy (dU/dT) table
dUdT_Table = pd.DataFrame({
    "SOC": model["SOC"],
    "dUdT": model["OCVrel"] / 1000  # Convert from mV/K to V/K
})

#Export tables as .CSV files
os.makedirs('OCP_data', exist_ok=True) 
os.makedirs('Entropy_data', exist_ok=True)
OCPTable.to_csv(f"OCP_data/{cellID}_OCP.csv", index=False)   # Uncomment line to save OCV table
dUdT_Table.to_csv(f"Entropy_data/{cellID}_dUdT.csv", index=False) # Uncomment line to save Entropy profile table

#Ploting the entropy profile C/3

plt.figure()
plt.plot(model["SOC"], model["OCVrel"])
plt.grid(True)

plt.title("Entropy figure from OCP data", fontsize=16)
plt.xlabel("SOC [-]", fontsize=14)
plt.ylabel("dU/dT [mV/K]", fontsize=14)

plt.xlim([0, 1])
plt.xticks(fontsize=14)
plt.ylim([0, model["OCVrel"].max() * 1.1])
plt.yticks(fontsize=14)

os.makedirs('Entropy_FIGURES', exist_ok=True)
plt.savefig(f'Entropy_FIGURES/{cellID}_Entropy_grid.png', dpi=300)

plt.show()




This concludes all the tutorial sessions in entropy estimation toolbox.

Thank you for your attention and interrest in the tool. 